In [2]:
import numpy as np
import cv2
import glob

In [3]:
PATTERN_SIZE = (9,6)
left_imgs = list(sorted(glob.glob('./stereo_calib_data/ittr7/*left.jpeg')))
right_imgs = list(sorted(glob.glob('./stereo_calib_data/ittr7/*right.jpeg')))
assert len(left_imgs)==len(right_imgs)

In [35]:
criteria = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)
left_pts, right_pts = [],[]
img_size = None
counter = 0
for left_img_path, right_img_path in zip(left_imgs, right_imgs):
    left_img = cv2.imread(left_img_path, cv2.IMREAD_GRAYSCALE)
    right_img = cv2.imread(right_img_path, cv2.IMREAD_GRAYSCALE)
    if img_size is None:
        img_size = (left_img.shape[1], left_img.shape[0])
    res_left, corners_left = cv2.findChessboardCorners(left_img, PATTERN_SIZE)
    res_right, corners_right = cv2.findChessboardCorners(right_img, PATTERN_SIZE)
    if res_left and res_right:
        corners_left = cv2.cornerSubPix(left_img, corners_left, (10,10), (-1,-1), criteria)
        corners_right = cv2.cornerSubPix(right_img, corners_right, (10,10), (-1,-1), criteria)
        
        left_pts.append(corners_left)
        right_pts.append(corners_right)
        print(counter)
        counter += 1

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14


In [36]:
pattern_points = np.zeros((np.prod(PATTERN_SIZE), 3), np.float32) 
pattern_points[:,:2] = np.indices(PATTERN_SIZE).T.reshape(-1, 2)
pattern_points = [pattern_points]*len(left_pts)
pattern_points

[array([[0., 0., 0.],
        [1., 0., 0.],
        [2., 0., 0.],
        [3., 0., 0.],
        [4., 0., 0.],
        [5., 0., 0.],
        [6., 0., 0.],
        [7., 0., 0.],
        [8., 0., 0.],
        [0., 1., 0.],
        [1., 1., 0.],
        [2., 1., 0.],
        [3., 1., 0.],
        [4., 1., 0.],
        [5., 1., 0.],
        [6., 1., 0.],
        [7., 1., 0.],
        [8., 1., 0.],
        [0., 2., 0.],
        [1., 2., 0.],
        [2., 2., 0.],
        [3., 2., 0.],
        [4., 2., 0.],
        [5., 2., 0.],
        [6., 2., 0.],
        [7., 2., 0.],
        [8., 2., 0.],
        [0., 3., 0.],
        [1., 3., 0.],
        [2., 3., 0.],
        [3., 3., 0.],
        [4., 3., 0.],
        [5., 3., 0.],
        [6., 3., 0.],
        [7., 3., 0.],
        [8., 3., 0.],
        [0., 4., 0.],
        [1., 4., 0.],
        [2., 4., 0.],
        [3., 4., 0.],
        [4., 4., 0.],
        [5., 4., 0.],
        [6., 4., 0.],
        [7., 4., 0.],
        [8., 4., 0.],
        [0

In [37]:
err, Kl, Dl, Kr, Dr, R, T, E, F = cv2.stereoCalibrate(
    pattern_points, left_pts, right_pts, None, None, None, None, img_size, flags=0
)

In [38]:
img_size

(1440, 1080)

In [39]:
print('Left camera:')
print(Kl)
print('Left camera distortion:')
print(Dl)
print('Right camera:')
print(Kr)
print('Right camera distortion:')
print(Dr)
print('Rotation matrix:')
print(R)
print('Translation:')
print(T)

Left camera:
[[1.78138934e+03 0.00000000e+00 7.08329104e+02]
 [0.00000000e+00 1.78323983e+03 5.42157147e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Left camera distortion:
[[-0.47244143  0.24769351  0.00411063  0.00256872 -0.12834181]]
Right camera:
[[1.77873195e+03 0.00000000e+00 7.20399078e+02]
 [0.00000000e+00 1.78246002e+03 5.40826659e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Right camera distortion:
[[-0.4390513  -0.06813465  0.00619669  0.00315972  0.85477487]]
Rotation matrix:
[[ 0.99939972  0.01880636  0.02909505]
 [-0.01895056  0.99980943  0.00468844]
 [-0.02900134 -0.00523699  0.99956565]]
Translation:
[[-4.55181673]
 [ 0.09878655]
 [ 0.07510906]]


In [42]:
img_l = cv2.imread(glob.glob('./stereo_calib_data/ittr7/*left.jpeg')[0])
ud_img_l = cv2.undistort(img_l, Kl, Dl)

img_r = cv2.imread(glob.glob('./stereo_calib_data/ittr7/*right.jpeg')[0])
ud_img_r = cv2.undistort(img_r, Kl, Dl)



cv2.imwrite("undistorted_l.jpeg", ud_img_l)
cv2.imwrite("distorted_l.jpeg", img_l)

cv2.imwrite("undistorted_r.jpeg", ud_img_r)
cv2.imwrite("distorted_r.jpeg", img_r)



True

In [41]:
np.save('./calibration_result/stereo.npy', {'Kl': Kl, 'Dl':Dl, 'Kr':Kr, 'Dr': Dr, 'R':R, 'T':T, 'E':E, 'F':F, 'img_size':img_size, 'left_pts': left_pts, 'right_pts':right_pts})